> **Versión corregida (v2.13, línea v1 de despliegue, ejecución verificada)** — generada a partir de `notebooks/14_documentacion_crisp_dm_y_reproducibilidad.ipynb` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de inicialización: se sustituyen la detección/montaje de Drive y el clonado de `main` por el bootstrap portable (`PROJECT_ROOT`/`REPO_ROOT`, rama `feature/entorno-capacidad-v2`).
> - Renombrado semántico: el antiguo `PROJECT_ROOT` (copia del repositorio) pasa a `REPO_ROOT`; `PERSISTENT_ROOT` se conserva como raíz de la línea v1 (= `PROJECT_ROOT`, o `PROJECT_ROOT/prueba_linea_v1` en modo prueba).
> - Corrige un error de sintaxis del original: `policy_table.rename(columns={...)` no cerraba la llave del diccionario (`SyntaxError`), por lo que el notebook no podía terminar.
> - Corrige un `NameError` del original: `{selected_metrics := ...:.3f}` dentro del f-string se interpretaba como especificador de formato; se usa directamente `selected_row.get(...)`.
> - Instala `tabulate` solo si falta (requerido por `DataFrame.to_markdown`).
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña, pocos pasos, salida en `prueba_linea_v1/`).


# PIDA-RL Diabetes — 14. Documentación CRISP-DM y reproducibilidad

Este notebook genera documentación estructurada del Proyecto Integrador de Dominio Autónomo (PIDA), siguiendo CRISP-DM. Consolida el problema, los datos, el simulador, modelos, evaluación, despliegue, resultados, limitaciones y requisitos de uso responsable.

> **Alcance:** los resultados provienen de un entorno de simulación construido con datos públicos y supuestos explícitos. No representan eficacia clínica, causalidad ni validación para uso institucional autónomo.


## 1. Propósito

El notebook crea documentos de evidencia para la entrega académica, el repositorio y la revisión metodológica. No reentrena modelos ni modifica resultados.

Productos generados:

- Reporte CRISP-DM en Markdown.
- Resumen ejecutivo en Markdown.
- Manifiesto de reproducibilidad en JSON.
- Inventario de artefactos y resultados en CSV.
- Extracto de resultados para documentación en CSV.


## 2. Inicialización


In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'feature/entorno-capacidad-v2')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

# --- Línea v1 de despliegue (entorno autocontenido) ---
# En modo prueba se escribe en una carpeta aparte para no sobrescribir resultados completos ni el dashboard publicado.
LINEA_V1_ROOT = PROJECT_ROOT / 'prueba_linea_v1' if MODO_PRUEBA else PROJECT_ROOT
PERSISTENT_ROOT = LINEA_V1_ROOT
COHORTE_CANONICA = PROJECT_ROOT / 'data' / 'processed' / 'cohorte_diabetes.csv'
print('Raíz de la línea v1 (PERSISTENT_ROOT):', PERSISTENT_ROOT)

# --- Celda original (sin montaje de Drive ni clonado fijo) ---
from pathlib import Path
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd

TABLES_DIR = PERSISTENT_ROOT / 'results' / 'tables'
FIGURES_DIR = PERSISTENT_ROOT / 'results' / 'figures'
METADATA_DIR = PERSISTENT_ROOT / 'data' / 'metadata'
DASHBOARD_DIR = PERSISTENT_ROOT / 'dashboard_streamlit'
DASHBOARD_DATA_DIR = PERSISTENT_ROOT / 'dashboard_data'
DOCUMENTATION_DIR = PERSISTENT_ROOT / 'documentation'

for directory in [DOCUMENTATION_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_ROOT)
print('Repositorio:', REPO_ROOT)
print('Resultados:', TABLES_DIR)
print('Documentación:', DOCUMENTATION_DIR)

# DataFrame.to_markdown requiere tabulate (incluido en Colab; se instala si falta).
if importlib.util.find_spec('tabulate') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabulate'], check=True)


## 3. Cargar evidencia final

Se utilizan los resultados compactos generados en los notebooks 09, 10 y 11.


In [ ]:
INPUT_FILES = {
    'ranking_tecnico': TABLES_DIR / 'ranking_tecnico_modelos_v1.csv',
    'tabla_decision': TABLES_DIR / 'tabla_decision_final_modelos_v1.csv',
    'conclusiones': TABLES_DIR / 'conclusiones_finales_pida_v1.csv',
    'equidad_riesgo': TABLES_DIR / 'equidad_por_riesgo_modelos_v1.csv',
    'brechas_riesgo': TABLES_DIR / 'brechas_por_riesgo_modelos_v1.csv',
    'alto_riesgo_semilla': TABLES_DIR / 'alto_riesgo_por_semilla_modelos_v1.csv',
    'protocolo_evaluacion': METADATA_DIR / 'protocolo_evaluacion_modelos_v1.json',
    'protocolo_equidad': METADATA_DIR / 'protocolo_equidad_riesgo_v3_1.json',
    'protocolo_final': METADATA_DIR / 'protocolo_visualizaciones_conclusiones_v1.json',
    'dashboard_config': DASHBOARD_DATA_DIR / 'configuracion_dashboard_v1.json',
}

for name, path in INPUT_FILES.items():
    print(f'{name:24s} | existe: {path.is_file()} | bytes: {path.stat().st_size if path.is_file() else 0}')

missing_files = [str(path) for path in INPUT_FILES.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Faltan insumos de los notebooks 09 a 11:\n' + '\n'.join(missing_files)
    )

df_ranking = pd.read_csv(INPUT_FILES['ranking_tecnico'])
df_decision = pd.read_csv(INPUT_FILES['tabla_decision'])
df_conclusions = pd.read_csv(INPUT_FILES['conclusiones'])
df_equidad_riesgo = pd.read_csv(INPUT_FILES['equidad_riesgo'])
df_brechas_riesgo = pd.read_csv(INPUT_FILES['brechas_riesgo'])
df_alto_riesgo_semilla = pd.read_csv(INPUT_FILES['alto_riesgo_semilla'])

with INPUT_FILES['protocolo_evaluacion'].open('r', encoding='utf-8') as file:
    protocol_evaluation = json.load(file)
with INPUT_FILES['protocolo_equidad'].open('r', encoding='utf-8') as file:
    protocol_equity = json.load(file)
with INPUT_FILES['protocolo_final'].open('r', encoding='utf-8') as file:
    protocol_final = json.load(file)
with INPUT_FILES['dashboard_config'].open('r', encoding='utf-8') as file:
    dashboard_config = json.load(file)

print('Evidencia final cargada correctamente.')


## 4. Validar resultados y extraer métricas


In [ ]:
expected_policies = {
    'aleatoria',
    'reglas_riesgo',
    'reglas_capacidad',
    'dqn_v1',
    'q_learning_v1',
    'ppo_v1',
}

for name, frame in {
    'ranking': df_ranking,
    'decisión': df_decision,
    'equidad por riesgo': df_equidad_riesgo,
    'brechas por riesgo': df_brechas_riesgo,
    'alto riesgo por semilla': df_alto_riesgo_semilla,
}.items():
    if 'politica' not in frame.columns:
        raise KeyError(f'El archivo de {name} no contiene la columna politica.')
    found_policies = set(frame['politica'].astype(str).unique())
    missing_policies = expected_policies - found_policies
    if missing_policies:
        raise ValueError(f'Faltan políticas en {name}: {sorted(missing_policies)}')

selected_policy = dashboard_config.get('selected_policy')
if selected_policy not in expected_policies:
    raise ValueError(f'La política seleccionada no es válida: {selected_policy}')

selected_row = df_decision.loc[df_decision['politica'] == selected_policy]
if selected_row.empty:
    raise ValueError(f'No hay resultados finales para la política seleccionada: {selected_policy}')
selected_row = selected_row.iloc[0]

selected_high_risk = df_alto_riesgo_semilla.loc[
    df_alto_riesgo_semilla['politica'] == selected_policy
]

if selected_high_risk.empty:
    raise ValueError(f'No hay evaluación por semilla para alto riesgo en {selected_policy}')

high_risk_mean = float(selected_high_risk['cobertura_porcentaje'].mean())
high_risk_std = float(selected_high_risk['cobertura_porcentaje'].std())

print('Política seleccionada:', selected_policy)
print('Estado de recomendación:', dashboard_config.get('recommendation_status'))
print('Cobertura promedio de alto riesgo:', round(high_risk_mean, 3))
print('Desviación entre semillas:', round(high_risk_std, 3))
display(df_decision.sort_values('ranking_tecnico'))


## 5. Inventario de reproducibilidad

Se registra la presencia, tamaño y hash SHA-256 de los artefactos compactos utilizados por el proyecto. Los hashes facilitan verificar que los documentos se generaron con una versión específica de los resultados.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for block in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

artifact_paths = {
    **INPUT_FILES,
    'figura_comparacion_final': FIGURES_DIR / 'comparacion_final_politicas_v1.png',
    'figura_equidad_riesgo': FIGURES_DIR / 'equidad_cobertura_por_riesgo_v1.png',
    'dashboard_app': DASHBOARD_DIR / 'app.py',
    'dashboard_requirements': DASHBOARD_DIR / 'requirements.txt',
    'dashboard_readme': DASHBOARD_DIR / 'README.md',
}

inventory_records = []
for artifact_name, path in artifact_paths.items():
    exists = path.is_file()
    inventory_records.append({
        'artefacto': artifact_name,
        'ruta': str(path),
        'existe': exists,
        'bytes': path.stat().st_size if exists else 0,
        'sha256': sha256_file(path) if exists else None,
    })

df_inventory = pd.DataFrame(inventory_records)
display(df_inventory)

if not df_inventory['existe'].all():
    missing_artifacts = df_inventory.loc[~df_inventory['existe'], 'artefacto'].tolist()
    raise FileNotFoundError('Faltan artefactos requeridos: ' + ', '.join(missing_artifacts))


## 6. Generar reporte CRISP-DM

El reporte incorpora valores extraídos de los archivos de resultados, con limitaciones explícitas.


In [ ]:
policy_table = (
    df_decision.sort_values('ranking_tecnico')[
        [
            'ranking_tecnico',
            'politica',
            'clasificacion',
            'puntaje_tecnico',
            'recompensa_media',
            'bajo',
            'medio',
            'alto',
            'eventos_adversos_medios',
            'recursos_usados_medios',
            'excesos_capacidad_medios',
            'respeta_capacidad_en_promedio',
        ]
    ]
    .copy()
)

policy_table['respeta_capacidad_en_promedio'] = policy_table['respeta_capacidad_en_promedio'].map({True: 'Sí', False: 'No'})
policy_table = policy_table.rename(columns={
    'ranking_tecnico': 'Ranking',
    'politica': 'Política',
    'clasificacion': 'Tipo',
    'puntaje_tecnico': 'Puntaje técnico',
    'recompensa_media': 'Recompensa media',
    'bajo': 'Cobertura bajo (%)',
    'medio': 'Cobertura medio (%)',
    'alto': 'Cobertura alto (%)',
    'eventos_adversos_medios': 'Eventos adversos medios',
    'recursos_usados_medios': 'Recursos medios',
    'excesos_capacidad_medios': 'Excesos de capacidad',
    'respeta_capacidad_en_promedio': 'Respeta capacidad',
})

policy_table_markdown = policy_table.to_markdown(index=False, floatfmt='.3f')

conclusions_markdown = '\n'.join(
    f'- **{row["tipo"].replace("_", " ").capitalize()}:** {row["contenido"]}'
    for _, row in df_conclusions.iterrows()
)

report_markdown = f'''# PIDA-RL Diabetes México

## Documentación CRISP-DM y reproducibilidad

**Versión del documento:** 1.0  \n**Fecha de generación (UTC):** {datetime.now(timezone.utc).isoformat()}  \n**Política de referencia en simulación:** `{selected_policy}`  \n**Estado:** `{dashboard_config.get('recommendation_status')}`

> Este documento describe un prototipo de aprendizaje por refuerzo evaluado en un entorno de simulación. No es evidencia de efectividad clínica, no opera con expedientes reales y no debe utilizarse para asignar seguimiento clínico a pacientes reales.

---

## 1. Business Understanding

### Problema
Los programas de seguimiento de personas con diabetes enfrentan capacidad limitada para llamadas, mensajes y teleorientación. El problema consiste en asignar de manera adaptativa la intensidad de seguimiento, priorizando perfiles de mayor riesgo sin exceder los recursos operativos disponibles.

### Objetivo analítico
Construir un entorno de simulación mensual compatible con Gymnasium y comparar políticas de referencia con agentes de aprendizaje por refuerzo para maximizar la recompensa definida por el simulador, elevar la cobertura de alto riesgo y controlar el uso de recursos.

### Pregunta de decisión
Bajo los supuestos del simulador, ¿qué política ofrece la mejor combinación de rendimiento, priorización de alto riesgo, capacidad operativa y estabilidad entre semillas?

### Criterio de éxito
La selección técnica utiliza un ranking compuesto con estos pesos: recompensa media 35%, cobertura de alto riesgo 30%, eventos adversos simulados 15%, cumplimiento de capacidad 10% y eficiencia de recursos 10%.

---

## 2. Data Understanding

### Alcance de los datos
El prototipo usa datos públicos y una cohorte/entorno de simulación. Los resultados disponibles para evaluación contienen agregaciones por política, semilla, paso, mes y categoría de riesgo.

### Variables evaluadas
Las trazas agregadas incorporan, entre otras, política, semilla, paso, mes, nivel y categoría de riesgo, acción, acción aceptada, contacto aceptado, evento adverso, recompensa, costo real y exceso de capacidad.

### Limitaciones de datos
- Las trazas agregadas de evaluación no contienen identificador individual, sexo, edad ni grupo etario.
- Por tanto, la auditoría producida evalúa distribución operacional **por nivel de riesgo**, no equidad demográfica por sexo o edad.
- Los datos públicos y los parámetros de simulación no sustituyen datos clínicos institucionales autorizados.

---

## 3. Data Preparation

### Representación de riesgo
La evaluación clasifica los estados en riesgo bajo, medio y alto. Esta estratificación se utiliza para comparar la cobertura de seguimiento y comprobar que la política no oculte falta de atención al grupo de mayor prioridad.

### Restricción operativa
El entorno registra recursos usados y acciones que exceden la capacidad. Una política puede intentar una acción inviable; el indicador de exceso permite evaluar si anticipa el límite de recursos.

### Reproducibilidad
Las evaluaciones emplean las semillas {protocol_evaluation.get('seeds', [42, 123, 2026])} y {protocol_evaluation.get('steps_per_episode', 'N/D')} pasos por episodio, de acuerdo con el protocolo exportado.

---

## 4. Modeling

### Políticas comparadas
- `aleatoria`: línea base sin priorización estructurada.
- `reglas_riesgo`: política basada en reglas de riesgo.
- `reglas_capacidad`: política basada en riesgo con atención explícita al límite de capacidad.
- `q_learning_v1`: agente tabular de Q-learning.
- `dqn_v1`: agente Deep Q-Network.
- `ppo_v1`: agente Proximal Policy Optimization.

### Interpretación del agente
Los agentes aprendidos se comparan con reglas de referencia interpretables. La presencia de una política de reglas competitiva es un resultado informativo: el valor añadido de Deep RL debe justificarse frente a alternativas simples, auditables y compatibles con la capacidad operativa.

---

## 5. Evaluation

### Resultados consolidados

{policy_table_markdown}

### Resultado principal
Bajo el protocolo actual, la política de referencia es `{selected_policy}`. Su puntaje técnico es {selected_row.get('puntaje_tecnico', np.nan):.3f}; la cobertura media de alto riesgo es {selected_row.get('alto', np.nan):.3f}%; los excesos promedio de capacidad son {selected_row.get('excesos_capacidad_medios', np.nan):.3f}; y la desviación de cobertura de alto riesgo entre semillas es {high_risk_std:.3f} puntos porcentuales.

### Equidad operacional por riesgo
La brecha absoluta por riesgo se interpreta como la diferencia entre la mayor y menor cobertura observada en los niveles bajo, medio y alto. La diferencia alto–bajo permite identificar la dirección de priorización. Una diferencia positiva puede ser coherente con el objetivo clínico del simulador, siempre que no implique abandono sistemático de los grupos de riesgo bajo o medio.

### Hallazgos exportados
{conclusions_markdown}

---

## 6. Deployment

### Dashboard
El proyecto incluye un dashboard Streamlit que carga resultados agregados compactos desde `dashboard_data/`. El dashboard presenta resumen ejecutivo, comparación de políticas, cobertura por riesgo, estabilidad entre semillas y limitaciones.

### Formas de visualización
- Vista temporal desde Google Colab mediante Streamlit y ngrok.
- Despliegue persistente desde GitHub mediante Streamlit Community Cloud.

### Restricciones de despliegue
El dashboard no debe recibir ni publicar datos personales, expedientes clínicos, trazas individuales identificables, tokens o secretos.

---

## 7. Conclusiones y limitaciones

### Conclusión técnica
`{selected_policy}` es una candidata operativa de referencia **solo dentro del simulador**. La conclusión depende de la función de recompensa, la dinámica de transición, la definición de riesgo, la capacidad modelada, las políticas comparadas y las ponderaciones del ranking.

### Limitaciones críticas
- No hay validación clínica prospectiva ni datos institucionales reales autorizados.
- Los eventos adversos, contactos y costos son resultados del simulador, no desenlaces clínicos observados.
- No se puede inferir desempeño causal en pacientes reales.
- La auditoría demográfica por sexo y edad requiere trazas individuales que incluyan tales atributos y un identificador sintético o institucional autorizado.
- Cualquier uso futuro debe mantener supervisión humana, trazabilidad, monitoreo y mecanismos de apelación/revisión.

### Hoja de ruta
1. Validar supuestos del simulador con expertos clínicos y operativos.
2. Acordar una función de recompensa trazable y clínicamente defendible.
3. Evaluar con datos institucionales autorizados, controles de privacidad y gobernanza.
4. Generar trazas individuales minimizadas para auditoría demográfica y de subgrupos.
5. Realizar validación prospectiva con supervisión humana antes de considerar integración operativa.

---

## 8. Artefactos de reproducibilidad

Los hashes SHA-256 y rutas de los insumos se encuentran en `inventario_artefactos_reproducibilidad_v1.csv` y `manifiesto_reproducibilidad_v1.json`.
'''

REPORT_PATH = DOCUMENTATION_DIR / 'reporte_crisp_dm_pida_rl_diabetes_v1.md'
REPORT_PATH.write_text(report_markdown, encoding='utf-8')

print('Reporte CRISP-DM generado:', REPORT_PATH)
print('Bytes:', REPORT_PATH.stat().st_size)


## 7. Generar resumen ejecutivo


In [ ]:
executive_summary = f'''# Resumen ejecutivo — PIDA-RL Diabetes México

## Problema

Los programas de seguimiento de diabetes tienen recursos limitados para contactar, orientar y dar continuidad a las personas. Este PIDA evalúa políticas que asignan seguimiento adaptativamente dentro de un simulador, priorizando el riesgo y respetando una capacidad operativa modelada.

## Enfoque

Se compararon tres políticas de referencia (`aleatoria`, `reglas_riesgo`, `reglas_capacidad`) y tres enfoques de aprendizaje por refuerzo (`q_learning_v1`, `dqn_v1`, `ppo_v1`). La evaluación utilizó semillas {protocol_evaluation.get('seeds', [42, 123, 2026])} y métricas de recompensa, cobertura de alto riesgo, eventos adversos simulados, recursos y excesos de capacidad.

## Resultado principal

La política `{selected_policy}` fue identificada como candidata operativa de referencia en simulación. Alcanzó el primer lugar del ranking técnico, reportó {selected_row.get('excesos_capacidad_medios', np.nan):.3f} excesos promedio de capacidad y tuvo una cobertura media de alto riesgo de {selected_row.get('alto', np.nan):.3f}%.

## Interpretación

El resultado indica que una regla explícita y auditable que incorpora capacidad puede competir o superar a agentes de RL bajo los supuestos actuales. PPO fue el agente aprendido mejor posicionado en el ranking técnico y DQN destacó por respetar en promedio la restricción de capacidad.

## Limitaciones

Los resultados son simulados, no clínicos. La auditoría disponible es por nivel de riesgo; no se presentan conclusiones por sexo o edad porque las trazas de evaluación agregadas no incluyen atributos demográficos ni identificadores individuales.

## Recomendación

Utilizar el dashboard únicamente para demostración académica y análisis exploratorio. Antes de cualquier uso institucional se requieren datos autorizados, validación clínica, revisión ética, auditoría demográfica, trazabilidad, monitoreo y supervisión humana.
'''

EXECUTIVE_PATH = DOCUMENTATION_DIR / 'resumen_ejecutivo_pida_rl_diabetes_v1.md'
EXECUTIVE_PATH.write_text(executive_summary, encoding='utf-8')

print('Resumen ejecutivo generado:', EXECUTIVE_PATH)


## 8. Exportar manifiesto e inventario


In [ ]:
INVENTORY_PATH = DOCUMENTATION_DIR / 'inventario_artefactos_reproducibilidad_v1.csv'
RESULTS_EXTRACT_PATH = DOCUMENTATION_DIR / 'extracto_resultados_finales_v1.csv'
MANIFEST_PATH = DOCUMENTATION_DIR / 'manifiesto_reproducibilidad_v1.json'

df_inventory.to_csv(INVENTORY_PATH, index=False, encoding='utf-8-sig')

results_extract_columns = [
    'ranking_tecnico',
    'politica',
    'clasificacion',
    'puntaje_tecnico',
    'recompensa_media',
    'cobertura_alto_riesgo_media',
    'bajo',
    'medio',
    'alto',
    'brecha_absoluta_riesgo_pp',
    'brecha_alto_menos_bajo_pp',
    'eventos_adversos_medios',
    'recursos_usados_medios',
    'excesos_capacidad_medios',
    'respeta_capacidad_en_promedio',
    'candidata_operativa',
]

available_results_columns = [column for column in results_extract_columns if column in df_decision.columns]
df_results_extract = df_decision[available_results_columns].sort_values('ranking_tecnico').copy()
df_results_extract.to_csv(RESULTS_EXTRACT_PATH, index=False, encoding='utf-8-sig')

git_commit = subprocess.run(
    ['git', 'rev-parse', 'HEAD'],
    text=True,
    capture_output=True,
    check=False,
).stdout.strip() or None

git_branch = subprocess.run(
    ['git', 'branch', '--show-current'],
    text=True,
    capture_output=True,
    check=False,
).stdout.strip() or None

manifest = {
    'project': 'PIDA-RL Diabetes México',
    'document_version': '1.0',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'selected_policy': selected_policy,
    'recommendation_status': dashboard_config.get('recommendation_status'),
    'evaluation_seeds': protocol_evaluation.get('seeds', [42, 123, 2026]),
    'steps_per_episode': protocol_evaluation.get('steps_per_episode'),
    'ranking_weights': protocol_evaluation.get('ranking_weights'),
    'audit_scope': protocol_equity.get('audit_scope'),
    'not_available_in_source_data': protocol_equity.get('not_available_in_source_data'),
    'repository': {
        'url': REPO_URL,
        'branch': git_branch,
        'commit': git_commit,
    },
    'runtime': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
    },
    'artifacts': df_inventory.to_dict(orient='records'),
    'limitations': [
        'Resultados generados en un simulador con datos públicos y supuestos explícitos.',
        'No existe validación clínica prospectiva ni causal sobre pacientes reales.',
        'La auditoría disponible es por nivel de riesgo y no incluye sexo o edad.',
        'El dashboard es demostrativo y no debe utilizarse para decisiones clínicas individuales.',
    ],
}

with MANIFEST_PATH.open('w', encoding='utf-8') as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

for path in [REPORT_PATH, EXECUTIVE_PATH, INVENTORY_PATH, RESULTS_EXTRACT_PATH, MANIFEST_PATH]:
    print(path.name, '| existe:', path.is_file(), '| bytes:', path.stat().st_size if path.is_file() else 0)

print('Documentación CRISP-DM y manifiesto de reproducibilidad completados.')


## Resultado esperado

El notebook debe terminar con:

```text
Documentación CRISP-DM y manifiesto de reproducibilidad completados.
```

Los documentos quedan en `PIDA-RL-Diabetes/documentation/`. Para la entrega final, el reporte CRISP-DM y el resumen ejecutivo pueden versionarse en GitHub junto con el dashboard, siempre revisando que no contengan datos sensibles.
